# Goal-Conditioned Analog AI Sizing (V9) - Kaggle Version
V9 Features (Architecture Fix):
- Rich 15D Observation: design params + performance + targets
- Direct Parameter Output (no delta-based movement)
- Absolute Normalized Reward (no delta-based reward)
- V8 Hierarchy of Needs cost function preserved
- Training: 500,000 steps

## Setup Instructions
1. **Upload Code:** Upload the `analog-ai-code-v9.zip` (from the V9_Trainer folder) as a Kaggle Dataset.
2. **Google Drive Quota Fix:** Make sure your LUT Google Drive IDs are correct.
3. Click **Save Version -> Save & Run All (Commit)**.

In [ ]:
!pip install stable-baselines3[extra] gymnasium numpy scipy tensorboard gdown

In [ ]:
import os
import sys

WORKING_DIR = '/kaggle/working/'
print("Finding LUTs from previous kernel output...")
nch_path = None
pch_path = None

for root, dirs, files in os.walk('/kaggle/input/ota-rl-model-v9/'):
    for file in files:
        if file == 'TSMC_fast_65nm_nch.pkl':
            nch_path = os.path.join(root, file)
        if file == 'TSMC_fast_65nm_pch.pkl':
            pch_path = os.path.join(root, file)

if nch_path and pch_path:
    print(f"Found NCH: {nch_path}")
    print(f"Found PCH: {pch_path}")
else:
    print("ERROR: Could not find LUTs in /kaggle/input/")

In [ ]:
import os
import sys
import gdown
import zipfile

# 1. Find the project root from Kaggle dataset
dataset_path = '/kaggle/input'
project_root = None

for root, dirs, files in os.walk(dataset_path):
    if 'core' in dirs and 'circuits' in dirs and 'v10' in root.lower():
        project_root = root
        break

if project_root is None:
    for root, dirs, files in os.walk(dataset_path):
        if 'core' in dirs and 'circuits' in dirs:
            project_root = root
            break

if project_root:
    print(f'Found project root at: {project_root}')
    sys.path.append(project_root)
else:
    print('Error: Could not find project root in dataset.')

In [ ]:
from tech_luts.lut_utils import LUT
from core.device_model import DeviceModel
from circuits.ota5t import OTA5T
from optimizer.rl_environment import OTA5tGymEnv

print("Loading LUTs into memory...")

nch = LUT(nch_path)
pch = LUT(pch_path)

dm = DeviceModel(nch, pch)
ota = OTA5T(dm, vdd=1.2)
print("Physics Engine Ready!")

In [ ]:
bounds = [
    (60e-9, 1.5e-6),  # L1
    (5.0, 25.0),      # gmid1
    (60e-9, 1.5e-6),  # L3
    (5.0, 25.0),      # gmid3
    (10e-6, 500e-6)   # Itail
]

env = OTA5tGymEnv(ota, bounds=bounds, max_steps=200)

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback

checkpoint_callback = CheckpointCallback(
    save_freq=100000, 
    save_path=os.path.join(WORKING_DIR, 'checkpoints'),
    name_prefix='kaggle_ppo_model_v10'
)

In [ ]:
tb_log_dir = os.path.join(WORKING_DIR, 'ppo_ota_tensorboard')

# V10: Larger network (256x256) to handle the richer 15D observation space.
# Lower learning rate for stability with direct parameter output.
policy_kwargs = dict(net_arch=[256, 256])
model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0001, batch_size=512, n_steps=2048, policy_kwargs=policy_kwargs, tensorboard_log=tb_log_dir)

print("Starting 500,000 Steps Training on Kaggle (V10 - Architecture Fix)...")
model.learn(total_timesteps=500000, callback=checkpoint_callback)

model_save_path = 'universal_ppo_agent_65nm_v10'
model.save(model_save_path)
print(f'Model saved to {model_save_path}.zip')